# Colab 19 — Diagnostic: is high-similarity *value* present-but-unread, or absent?

The deployed encoder retrieves high-similarity neighbours essentially perfectly on
AA (ranking is solved), but the **predicted similarity saturates** inside the high
band — pairs with true `normLev` of 0.72 and 0.98 both map to ~0.8. That is the
documented consequence of the 3-bin CE head: every pair with `normLev >= 0.70` is
the *same class*, so no gradient separates 0.72 from 0.98.

This notebook does **not** change the training design. It only *measures*, to decide
one thing before we spend effort on a `lambda*MSE` head:

> Is the within-high-band value information **present in the embedding but unread**
> (saturated onto a narrow but monotone range we can straighten with a recalibration),
> or **genuinely absent** (CE collapsed it, needs finer supervision)?

**Method (value).** Restricted to high-sim pairs only (`>= 0.70` — see note), fit an
**isotonic** (best monotone, least-squares) map predicted-L2-sim -> true `normLev`,
cross-validated. Report **Spearman rho** (order preserved?) and the **isotonic OOF
RMSE in normLev units** (after removing any monotone distortion, how tight is the
recoverable value?). Small RMSE -> value present, only the readout saturates -> cheap
fix. Fat RMSE -> value absent -> `lambda*MSE` warranted. **RMSE is the decision number.**

**Method (ranking, set-based).** Designated-single-partner retrieval is the wrong
question in a crowded alphabet. We use an **exact-Levenshtein oracle**: for each query,
the ground-truth neighbour set is every pool protein with exact `normLev >= 0.70`. We
report **median |true_set|** (crowding), and **frac-achievable@k =
retrieved / min(k, |true_set|)** (1.0 = fills every achievable top-k slot with a true-high
neighbour — the oracle high-set *ceiling*, not identical ordering to exact Lev).

> **Why `>= 0.70` only for value.** A whole-range Spearman/isotonic is dominated by the
> easy far->high gradient (trivially separates 0.1 from 0.9), reading ~0.9+ while hiding
> whether the *high band* is resolved. Every value statistic here is on `>= 0.70` pairs.

Encoders + recipe are lifted unchanged from **colab17b** (AA 20-letter, SS 3-letter),
evaluated as a full **2x3** matrix ({AA,SS} encoders x {AA,SS,3Di} feeds). RNG is
threaded for determinism; the synthetic diagnostic set is frozen to a committable CSV.
The full-alphabet / natural-language generality probe is deferred to **colab20**; a true
masked-adaptive-pooling ablation (current pooling averages over zeroed PAD buckets ->
length leak) is deferred to **colab21**.


## 1. Setup

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
DATA_DIR = '/content/thesis-edit-distance-nn/sampledata/cath'
import os
for f in ['cath_s20_train70.csv.gz', 'cath_s20_test30.csv.gz', 'cath_eval.csv.gz', 'cath_s20_3di.csv.gz']:
    p = os.path.join(DATA_DIR, f)
    print(f'{"OK" if os.path.exists(p) else "MISSING":<8} {p}')

In [ ]:
!pip install torch torchvision rapidfuzz scikit-learn scipy --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import KFold, GroupKFold
from scipy.stats import spearmanr
from rapidfuzz.distance import Levenshtein as RFLevenshtein
from rapidfuzz.process import cdist as rf_cdist

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Constants (encoder recipe identical to colab17b)

In [ ]:
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'
SS_ALPHABET = 'HLS'
VOCAB_SIZE = len(AA_ALPHABET) + 1
PAD_IDX = len(AA_ALPHABET)
AA_SET = set(AA_ALPHABET)
SS_SET = set(SS_ALPHABET)
CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}

MIN_LEN = 50
MAX_LEN_SEQ = 200
MAX_LEN = 200

N_TRAIN = 30000
NUM_EPOCHS = 30
BATCH_SIZE = 128
K = 16

BAND_LOW_AA = 0.30
BAND_LOW_SS = 0.56
BAND_HIGH   = 0.70

BIN_NAMES = ['far', 'mid', 'high']

# Diagnostic-specific
N_SYNTH_HIGH = 5000          # synthetic high-sim AA pairs for the powered verdict
N_FOLDS = 5
N_BOOT = 1000
K_LIST = (1, 5, 10, 50)

# Reproducibility / freezing
SEED_TRAIN_AA = 42
SEED_TRAIN_SS = 43
SEED_SYNTH    = 12345
SYNTH_PATH = os.path.join(DATA_DIR, 'cath_synth_highsim_aa.csv.gz')
LOAD_FROZEN_SYNTH = True      # if the committed CSV exists, use it (pins the verdict)
DUMP_TRAIN_PAIRS = False      # set True to also WRITE the 30k training pairs to CSV (write-only)

print(f'AA encoder BAND_LOW={BAND_LOW_AA} | SS encoder BAND_LOW={BAND_LOW_SS} '
      f'| high-sim cutoff={BAND_HIGH}')

## 3. Helpers — Levenshtein, encode, perturb (RNG threaded for determinism)

In [ ]:
def norm_lev(a, b):
    L = max(len(a), len(b))
    return 1.0 if L == 0 else 1.0 - RFLevenshtein.distance(a, b) / L

def is_standard_aa(seq): return all(c in AA_SET for c in seq)
def is_standard_ss(seq): return all(c in SS_SET for c in seq)

def encode_pad(seq, max_len=MAX_LEN, pad_idx=PAD_IDX):
    idx = [CHAR_TO_IDX[c] for c in seq][:max_len]
    idx += [pad_idx] * (max_len - len(idx))
    return torch.tensor(idx, dtype=torch.long)

def perturb(seq, k, alphabet, rng, max_len=MAX_LEN):
    s = list(seq); abc = list(alphabet)
    for _ in range(k):
        if len(s) == 0: op = 'ins'
        elif len(s) >= max_len: op = rng.choice(['sub', 'del'])
        else: op = rng.choice(['sub', 'ins', 'del'])
        if op == 'sub':
            i = rng.integers(0, len(s))
            choices = [c for c in abc if c != s[i]]
            s[i] = rng.choice(choices)
        elif op == 'ins':
            i = rng.integers(0, len(s) + 1)
            s.insert(i, rng.choice(abc))
        elif op == 'del':
            i = rng.integers(0, len(s))
            del s[i]
    return ''.join(s)

def random_seq(alphabet, rng, min_len=MIN_LEN, max_len=MAX_LEN_SEQ):
    length = int(rng.integers(min_len, max_len + 1))
    return ''.join(rng.choice(list(alphabet), size=length))

def bin_idx_for(x, band_low):
    if x < band_low: return 0
    if x < BAND_HIGH: return 1
    return 2

def make_full_range_pairs(alphabet, n_pairs, rng):
    """t ~ U[0,1] -> roughly uniform coverage of normLev (far/mid/high all present)."""
    pairs = []
    attempts = 0; max_attempts = n_pairs * 4
    while len(pairs) < n_pairs and attempts < max_attempts:
        attempts += 1
        seed = random_seq(alphabet, rng)
        if len(seed) < MIN_LEN: continue
        L = len(seed)
        t = float(rng.uniform(0.0, 1.0))
        k = max(0, int(round((1.0 - t) * L)))
        other = perturb(seed, k, alphabet, rng)
        if 1 <= len(other) <= MAX_LEN:
            pairs.append((seed, other, norm_lev(seed, other)))
    return pairs

def make_high_sim_pairs(alphabet, n_pairs, rng, lo=BAND_HIGH):
    """High-band-ONLY pairs (>= lo) for the value-fidelity diagnostic."""
    pairs = []
    attempts = 0; max_attempts = n_pairs * 10
    while len(pairs) < n_pairs and attempts < max_attempts:
        attempts += 1
        seed = random_seq(alphabet, rng)
        if len(seed) < MIN_LEN: continue
        L = len(seed)
        t = float(rng.uniform(lo, 1.0))
        k = max(0, int(round((1.0 - t) * L)))
        other = perturb(seed, k, alphabet, rng)
        if 1 <= len(other) <= MAX_LEN:
            lab = norm_lev(seed, other)
            if lab >= lo:
                pairs.append((seed, other, lab))
    return pairs

## 4. Architecture + training (3-bin CE head, identical to colab17b)

In [ ]:
class SiameseEncoder(nn.Module):
    def __init__(self, K, vocab_size=VOCAB_SIZE, embed_dim=32,
                 hidden1=32, hidden2=64, out_dim=128, pad_idx=PAD_IDX):
        super().__init__()
        self.K = K; self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.conv1 = nn.Conv1d(embed_dim, hidden1, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(hidden1, hidden2, kernel_size=3, padding=1)
        self.pool  = nn.AdaptiveAvgPool1d(K)   # NOTE: averages over zeroed PAD buckets -> colab21 ablation
        self.fc    = nn.Linear(hidden2 * K, out_dim)

    def forward(self, x):
        mask = (x != self.pad_idx).float()
        e = self.embedding(x).permute(0, 2, 1)
        h = F.relu(self.conv1(e)); h = F.relu(self.conv2(h))
        h = h * mask.unsqueeze(1)
        h = self.pool(h).flatten(1)
        return F.normalize(self.fc(h), p=2, dim=1)


class SiameseClassifier(nn.Module):
    def __init__(self, K, embed_out=128, hidden_mlp=64, n_bins=3):
        super().__init__()
        self.encoder = SiameseEncoder(K)
        self.head = nn.Sequential(
            nn.Linear(embed_out, hidden_mlp), nn.LeakyReLU(0.01),
            nn.Linear(hidden_mlp, n_bins))

    def forward(self, a, b):
        return self.head(torch.abs(self.encoder(a) - self.encoder(b)))
    def encode(self, x):
        return self.encoder(x)


class PairDatasetCE(Dataset):
    def __init__(self, pairs, band_low):
        self.a = [encode_pad(a) for a, _, _ in pairs]
        self.b = [encode_pad(b) for _, b, _ in pairs]
        self.bins = torch.tensor([bin_idx_for(l, band_low) for _, _, l in pairs], dtype=torch.long)
    def __len__(self): return len(self.bins)
    def __getitem__(self, i): return self.a[i], self.b[i], self.bins[i]


def train_encoder(alphabet, band_low, label, train_seed, n_epochs=NUM_EPOCHS):
    torch.manual_seed(train_seed)
    rng = np.random.default_rng(train_seed)         # local RNG -> deterministic regardless of cell order
    print(f'\n===== Training {label} encoder (alphabet={alphabet}, band_low={band_low}, seed={train_seed}) =====')
    pairs = make_full_range_pairs(alphabet, N_TRAIN, rng)
    labels = np.array([p[2] for p in pairs])
    counts = {b: int(c) for b, c in zip(BIN_NAMES,
              np.bincount([bin_idx_for(l, band_low) for l in labels], minlength=3))}
    print(f'  {len(pairs)} pairs, normLev in [{labels.min():.3f}, {labels.max():.3f}], bins={counts}')
    if DUMP_TRAIN_PAIRS:
        pd.DataFrame(pairs, columns=['seq_a', 'seq_b', 'norm_lev']).to_csv(
            f'colab19_train_{label}.csv.gz', index=False)
    dl = DataLoader(PairDatasetCE(pairs, band_low), batch_size=BATCH_SIZE, shuffle=True)
    model = SiameseClassifier(K).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    model.train()
    for epoch in range(1, n_epochs + 1):
        tot = 0.0; nb = 0
        for a, b, y in dl:
            a, b, y = a.to(device), b.to(device), y.to(device)
            loss = crit(model(a, b), y)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item(); nb += 1
        if epoch % 5 == 0 or epoch == 1:
            print(f'  [{label}] epoch {epoch:3d}/{n_epochs} CE={tot/nb:.4f}')
    model.eval()
    return model

## 5. Data load — pool, eval set, 3Di per-pair score (from colab17b)

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_train70.csv.gz'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_test30.csv.gz'))
prot_df = pd.concat([train_df, test_df], ignore_index=True).drop_duplicates('domain_id')
prot_df = prot_df[prot_df['aa_seq'].apply(is_standard_aa)]
prot_df = prot_df[prot_df['ss_seq'].apply(is_standard_ss)]
prot_df = prot_df[prot_df['aa_seq'].str.len() == prot_df['ss_seq'].str.len()]
prot_df['aa_len'] = prot_df['aa_seq'].str.len()
RESCUED = {'4z0mC02', '3qkaE02'}
in_range = prot_df['aa_len'].between(MIN_LEN, MAX_LEN)
prot_df = prot_df[in_range | prot_df['domain_id'].isin(RESCUED)].reset_index(drop=True)

id_to_aa = dict(zip(prot_df['domain_id'], prot_df['aa_seq']))
id_to_ss = dict(zip(prot_df['domain_id'], prot_df['ss_seq']))
all_domains = list(prot_df['domain_id'])
print(f'Protein pool: {len(prot_df)}')

eval_df = pd.read_csv(os.path.join(DATA_DIR, 'cath_eval.csv.gz'))
seqs3 = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_3di.csv.gz'))
id_to_3di = dict(zip(seqs3['domain_id'], seqs3['3di'].astype(str)))

def pair_3di_score(a, b):
    if a in id_to_3di and b in id_to_3di:
        return norm_lev(id_to_3di[a], id_to_3di[b])
    return np.nan
eval_df['3di_score'] = [pair_3di_score(a, b) for a, b in zip(eval_df['domain_a'], eval_df['domain_b'])]

SCORE_COL = {'AA': 'aa_score', 'SS': 'ss_score', '3Di': '3di_score'}
LOOKUP    = {'AA': id_to_aa,  'SS': id_to_ss,  '3Di': id_to_3di}
for feed, sc in SCORE_COL.items():
    n = int((eval_df[sc] >= BAND_HIGH).sum())
    tag = 'powered' if n >= 200 else ('moderate' if n >= 30 else 'underpowered (n<30)')
    print(f'  {feed}-feed: {n} eval pairs >= {BAND_HIGH}  [{tag}]')

## 6. Train the two encoders (frozen afterwards — no design change)

In [ ]:
model_aa = train_encoder(AA_ALPHABET, BAND_LOW_AA, 'AA', SEED_TRAIN_AA)
model_ss = train_encoder(SS_ALPHABET, BAND_LOW_SS, 'SS', SEED_TRAIN_SS)

## 7. Diagnostic machinery — predicted L2-sim, isotonic (row + group CV), bootstrap

- `pred_sim_strings`: deployment score `1 - ||e_a - e_b|| / 2` for arbitrary pairs.
- `isotonic_rmse`: CV isotonic fit pred->true; `groups` enables connected-component
  held-out CV (no shared protein across folds) for natural pairs.
- `bootstrap_ci`: percentile CI for Spearman and (in-sample) isotonic RMSE.


In [ ]:
@torch.no_grad()
def encode_pool(model, seq_lookup, ids, batch_size=256):
    model.eval()
    valid = [d for d in ids if d in seq_lookup]
    out = []
    for i in range(0, len(valid), batch_size):
        x = torch.stack([encode_pad(seq_lookup[d]) for d in valid[i:i+batch_size]]).to(device)
        out.append(model.encoder(x))
    return (torch.cat(out, 0) if out else torch.empty(0)), valid

@torch.no_grad()
def pred_sim_strings(model, A, B, batch=512):
    model.eval(); sims = []
    for i in range(0, len(A), batch):
        xa = torch.stack([encode_pad(s) for s in A[i:i+batch]]).to(device)
        xb = torch.stack([encode_pad(s) for s in B[i:i+batch]]).to(device)
        d = torch.linalg.vector_norm(model.encoder(xa) - model.encoder(xb), dim=1)
        sims.append((1.0 - d / 2.0).cpu().numpy())
    return np.concatenate(sims) if sims else np.array([])

def component_groups(pairs):
    """Union-find -> connected-component id per pair (a & b share a component)."""
    parent = {}
    def find(x):
        parent.setdefault(x, x); root = x
        while parent[root] != root: root = parent[root]
        while parent[x] != root: parent[x], x = root, parent[x]
        return root
    for a, b in pairs:
        ra, rb = find(a), find(b)
        if ra != rb: parent[ra] = rb
    return [find(a) for a, b in pairs]

def isotonic_oof(true, pred, groups=None, n_folds=N_FOLDS, seed=SEED):
    """CV out-of-fold isotonic fit. Returns (OOF RMSE, OOF residual array)."""
    true = np.asarray(true, float); pred = np.asarray(pred, float); n = len(true)
    if n < n_folds * 2 or np.std(pred) == 0: return np.nan, None
    oof = np.full(n, np.nan)
    if groups is None:
        split = KFold(n_splits=n_folds, shuffle=True, random_state=seed).split(pred)
    else:
        g = np.asarray(groups); ng = len(np.unique(g))
        if ng < 2: return np.nan, None
        split = GroupKFold(n_splits=min(n_folds, ng)).split(pred, groups=g)
    for tr, te in split:
        ir = IsotonicRegression(out_of_bounds='clip').fit(pred[tr], true[tr])
        oof[te] = ir.predict(pred[te])
    resid = true - oof
    return float(np.sqrt(np.nanmean(resid ** 2))), resid

def isotonic_rmse(true, pred, groups=None, n_folds=N_FOLDS, seed=SEED):
    return isotonic_oof(true, pred, groups, n_folds, seed)[0]

def spearman_hi(true, pred):
    if len(true) < 4 or np.std(pred) == 0 or np.std(true) == 0: return np.nan
    return float(spearmanr(pred, true).correlation)

def bootstrap_rmse_ci(oof_resid, B=N_BOOT, seed=SEED):
    """95% CI for the OOF isotonic RMSE by resampling the OUT-OF-FOLD residuals.

    Same OOF quantity as the point estimate (NOT optimistic in-sample RMSE)."""
    if oof_resid is None: return (np.nan, np.nan)
    r = np.asarray(oof_resid, float); r = r[~np.isnan(r)]; n = len(r)
    if n < 8: return (np.nan, np.nan)
    brng = np.random.default_rng(seed)
    out = [float(np.sqrt(np.mean(r[brng.integers(0, n, n)] ** 2))) for _ in range(B)]
    return float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))

def bootstrap_spearman_ci(true, pred, B=N_BOOT, seed=SEED):
    true = np.asarray(true, float); pred = np.asarray(pred, float); n = len(true)
    if n < 8 or np.std(pred) == 0: return (np.nan, np.nan)
    brng = np.random.default_rng(seed); rhos = []
    for _ in range(B):
        idx = brng.integers(0, n, n); t, p = true[idx], pred[idx]
        if np.std(p) == 0 or np.std(t) == 0: continue
        rhos.append(spearmanr(p, t).correlation)
    return (float(np.percentile(rhos, 2.5)), float(np.percentile(rhos, 97.5))) if rhos else (np.nan, np.nan)

def fmt(x, d=3):
    return 'n/a' if (x is None or (isinstance(x, float) and np.isnan(x))) else f'{x:.{d}f}'

## 8. STEP A — Synthetic high-sim verdict (AA, well-powered)

The verdict for AA. Thousands of **synthetic** high-sim AA pairs (random seed + few
edits, `normLev >= 0.70`) — the only well-powered AA substrate (natural CATH has ~5).
In-distribution for the encoder, so it isolates the architecture's *capacity* to encode
within-band value, free of natural distribution-shift. The set is **frozen** to
`cath_synth_highsim_aa.csv.gz` (loaded if already committed) so the verdict is pinned.


In [ ]:
if LOAD_FROZEN_SYNTH and os.path.exists(SYNTH_PATH):
    sdf = pd.read_csv(SYNTH_PATH)
    synth = list(zip(sdf['seq_a'], sdf['seq_b'], sdf['norm_lev']))
    print(f'Loaded FROZEN synthetic high-sim set: {SYNTH_PATH} (n={len(synth)})')
else:
    rng_synth = np.random.default_rng(SEED_SYNTH)
    synth = make_high_sim_pairs(AA_ALPHABET, N_SYNTH_HIGH, rng_synth)
    pd.DataFrame(synth, columns=['seq_a', 'seq_b', 'norm_lev']).to_csv(SYNTH_PATH, index=False)
    print(f'Generated + wrote synthetic high-sim set -> {SYNTH_PATH} (n={len(synth)})')
    print('  (commit this file to sampledata/cath/ to freeze the verdict for the thesis.)')
    if len(synth) < N_SYNTH_HIGH:
        print(f'  WARNING: only {len(synth)}/{N_SYNTH_HIGH} pairs (sampler hit max_attempts)')

assert all(l >= BAND_HIGH for *_, l in synth), 'synthetic diagnostic set contains sub-0.70 pairs!'
print(f'  sanity: n={len(synth)}, min normLev={min(l for *_, l in synth):.3f} (>= {BAND_HIGH})')

sy_true = np.array([p[2] for p in synth])
sy_pred = pred_sim_strings(model_aa, [p[0] for p in synth], [p[1] for p in synth])

rmse_synth, resid_synth = isotonic_oof(sy_true, sy_pred)   # row-wise CV (pairs independent here)
rho_synth = spearman_hi(sy_true, sy_pred)
rho_lo, rho_hi = bootstrap_spearman_ci(sy_true, sy_pred)
rm_lo, rm_hi = bootstrap_rmse_ci(resid_synth)              # CI of the OOF RMSE (resamples OOF residuals)
print('\n--- AA SYNTHETIC high-sim verdict (>= 0.70) ---')
print(f'  pred L2-sim: mean={sy_pred.mean():.3f}  std={sy_pred.std():.3f}')
print(f'  Spearman rho      : {fmt(rho_synth)}   95% CI [{fmt(rho_lo)}, {fmt(rho_hi)}]')
print(f'  isotonic OOF RMSE : {fmt(rmse_synth)}   95% CI [{fmt(rm_lo)}, {fmt(rm_hi)}]  <-- DECISION NUMBER')
print('  (RMSE CI = bootstrap of out-of-fold residuals; same OOF quantity as the point estimate.)')

In [ ]:
# Sub-band breakdown: where inside the high band does saturation bite?
print(f'  {"band":<14}{"n":>7}{"pred_mean":>11}{"pred_std":>10}{"spearman":>10}')
for lo, hi in [(0.70, 0.80), (0.80, 0.90), (0.90, 1.001)]:
    m = (sy_true >= lo) & (sy_true < hi)
    if m.sum() >= 4:
        print(f'  [{lo:.2f},{hi:.2f})  {int(m.sum()):>7}{sy_pred[m].mean():>11.3f}'
              f'{sy_pred[m].std():>10.3f}{fmt(spearman_hi(sy_true[m], sy_pred[m])):>10}')

In [ ]:
# Scatter: predicted L2-sim vs true normLev + the isotonic recalibration curve.
# Natural high-AA pairs (n~5) overlaid as reference diamonds (NOT used in the verdict).
nat_aa = eval_df[(eval_df['aa_score'] >= BAND_HIGH)
                 & eval_df['domain_a'].isin(id_to_aa) & eval_df['domain_b'].isin(id_to_aa)]
na_true = nat_aa['aa_score'].values
na_pred = pred_sim_strings(model_aa, [id_to_aa[d] for d in nat_aa['domain_a']],
                                     [id_to_aa[d] for d in nat_aa['domain_b']])
ir_full = IsotonicRegression(out_of_bounds='clip').fit(sy_pred, sy_true)
grid = np.linspace(sy_pred.min(), sy_pred.max(), 200)

plt.figure(figsize=(7, 6))
plt.scatter(sy_pred, sy_true, s=4, alpha=0.12, label=f'synthetic high-sim (n={len(synth)})')
plt.plot(grid, ir_full.predict(grid), 'g-', lw=2, label='isotonic recalibration')
plt.plot([0.6, 1.0], [0.6, 1.0], 'r--', lw=1, label='y = x (perfect)')
if len(na_pred):
    plt.scatter(na_pred, na_true, s=90, c='k', marker='D', zorder=5,
                label=f'natural CATH high-AA (n={len(na_pred)}, ref only)')
plt.xlabel('predicted L2-sim'); plt.ylabel('true normLev')
plt.title('AA high-sim: predicted vs true (isotonic = recoverable value)')
plt.legend(); plt.tight_layout(); plt.savefig('colab19_aa_isotonic.png', dpi=120); plt.show()

## 9. STEP B — Full 2x3 matrix: value fidelity + set-based oracle retrieval

For each (encoder, feed) we measure **value** (isotonic RMSE on natural `>= 0.70` pairs,
row-wise + connected-component-held-out CV, with bootstrap CIs) and **ranking** against
an **exact-Levenshtein oracle**:

- `median_n_true` = median size of the true high-sim neighbour set (crowding — how many
  there *are*).
- `frac@k = retrieved / min(k, |true_set|)` = fraction of the achievable top-k true
  neighbours the encoder surfaces. **1.0 = matches the oracle high-set ceiling** (all
  achievable top-k slots are true-high neighbours — NOT the same ordering as exact Lev),
  comparable across alphabets despite crowding.

The **dissociation** for SS: value can be present (isotonic RMSE small) while ranking is
capped low only because the 3-letter pool is genuinely crowded (large `median_n_true`).
That separates "encoder can't encode SS magnitude" from "alphabet too coarse to retrieve in".


In [ ]:
# Oracle neighbour sets per feed (encoder-independent) — computed once, reused.
def feed_high_pairs(feed):
    sc, lk = SCORE_COL[feed], LOOKUP[feed]
    inpool = set(all_domains) & set(lk)   # in retrieval pool AND has a sequence in this alphabet
    sub = eval_df.dropna(subset=[sc])
    sub = sub[(sub[sc] >= BAND_HIGH)
              & sub['domain_a'].isin(inpool) & sub['domain_b'].isin(inpool)]
    return sub

def build_oracle_cache(feed):
    lk = LOOKUP[feed]
    pool_ids = [d for d in all_domains if d in lk]
    idx = {d: i for i, d in enumerate(pool_ids)}
    pool_seqs = [lk[d] for d in pool_ids]; lens = np.array([len(s) for s in pool_seqs])
    sub = feed_high_pairs(feed)
    q_ids = sorted({d for pr in sub[['domain_a', 'domain_b']].values for d in pr})
    if not q_ids:
        return {'feed': feed, 'pool_ids': pool_ids, 'idx': idx, 'q_ids': [], 'true_sets': {}}
    D = rf_cdist([lk[q] for q in q_ids], pool_seqs,
                 scorer=RFLevenshtein.distance, workers=-1).astype(float)
    true_sets = {}
    for r, q in enumerate(q_ids):
        qi = idx[q]; denom = np.maximum(lens, lens[qi]); denom[denom == 0] = 1
        osim = 1.0 - D[r] / denom; osim[qi] = -np.inf
        true_sets[q] = set(np.where(osim >= BAND_HIGH)[0].tolist())
    n_true = [len(true_sets[q]) for q in q_ids]
    print(f'  oracle[{feed}]: {len(q_ids)} queries, median |true_set|={int(np.median(n_true))}, '
          f'max={max(n_true)}')
    return {'feed': feed, 'pool_ids': pool_ids, 'idx': idx, 'q_ids': q_ids, 'true_sets': true_sets}

print('Building exact-Levenshtein oracle neighbour sets...')
ORACLE = {feed: build_oracle_cache(feed) for feed in ['AA', 'SS', '3Di']}

In [ ]:
def retrieval_encoder(model, feed, k_list=K_LIST):
    cache = ORACLE[feed]; lk = LOOKUP[feed]
    if not cache['q_ids']:
        return {'median_n_true': np.nan, 'n_q': 0, **{k: {'frac': np.nan, 'prec': np.nan} for k in k_list}}
    pool_emb, _ = encode_pool(model, lk, cache['pool_ids'])
    idx = cache['idx']
    acc = {k: {'frac': [], 'prec': [], 'ret': []} for k in k_list}; nts = []
    for q in cache['q_ids']:
        qi = idx[q]; ts = cache['true_sets'][q]; nt = len(ts); nts.append(nt)
        if nt == 0: continue
        esim = (1.0 - torch.linalg.vector_norm(pool_emb - pool_emb[qi], dim=1) / 2.0).cpu().numpy()
        esim[qi] = -np.inf
        order = np.argsort(-esim)
        for k in k_list:
            ret = len(set(order[:k].tolist()) & ts)
            acc[k]['ret'].append(ret); acc[k]['prec'].append(ret / k)
            acc[k]['frac'].append(ret / min(k, nt))
    out = {'median_n_true': float(np.median(nts)), 'n_q': len(nts)}
    for k in k_list:
        out[k] = {'frac': float(np.mean(acc[k]['frac'])) if acc[k]['frac'] else np.nan,
                  'prec': float(np.mean(acc[k]['prec'])) if acc[k]['prec'] else np.nan,
                  'med_ret': float(np.median(acc[k]['ret'])) if acc[k]['ret'] else np.nan}
    return out

ENCODERS = [('AA-enc', model_aa), ('SS-enc', model_ss)]
FEEDS = ['AA', 'SS', '3Di']
rows = []; per_pair = []
for enc_label, model in ENCODERS:
    for feed in FEEDS:
        sub = feed_high_pairs(feed); sc = SCORE_COL[feed]; lk = LOOKUP[feed]
        true = sub[sc].values
        pred = pred_sim_strings(model, [lk[d] for d in sub['domain_a']],
                                       [lk[d] for d in sub['domain_b']])
        groups = component_groups(sub[['domain_a', 'domain_b']].values.tolist()) if len(sub) else None
        rmse_row = isotonic_rmse(true, pred)
        rmse_grp, resid_grp = isotonic_oof(true, pred, groups=groups)
        rho = spearman_hi(true, pred)
        rlo, rhi = bootstrap_spearman_ci(true, pred)
        mlo, mhi = bootstrap_rmse_ci(resid_grp)   # CI on the trusted (group-OOF) RMSE
        ret = retrieval_encoder(model, feed)
        cell = f'{enc_label} x {feed}-feed'
        rows.append({'cell': cell, 'n_pairs': len(sub), 'pred_mean': float(np.mean(pred)) if len(pred) else np.nan,
                     'spearman': rho, 'rho_lo': rlo, 'rho_hi': rhi,
                     'iso_rmse_row': rmse_row, 'iso_rmse_grp': rmse_grp, 'rmse_lo': mlo, 'rmse_hi': mhi,
                     'median_n_true': ret['median_n_true'],
                     'frac10': ret[10]['frac'], 'prec10': ret[10]['prec'], 'frac50': ret[50]['frac']})
        for (a, b), t, p in zip(sub[['domain_a', 'domain_b']].values, true, pred):
            per_pair.append({'cell': cell, 'domain_a': a, 'domain_b': b, 'true': t, 'pred': p})
        print(f'{cell:<20} n={len(sub):>4}  rho={fmt(rho)}  iso_rmse(row/grp)={fmt(rmse_row)}/{fmt(rmse_grp)}'
              f'  med|true|={fmt(ret["median_n_true"],0)}  frac@10={fmt(ret[10]["frac"])}')

## 10. STEP C — Decision table + CSV exports

In [ ]:
def verdict(d):
    rmse = d['iso_rmse_grp'] if not np.isnan(d.get('iso_rmse_grp', np.nan)) else d.get('iso_rmse_row', np.nan)
    rho, n = d.get('spearman', np.nan), d.get('n_pairs', d.get('n', 0))
    if n < 10 or rmse is None or np.isnan(rmse): return 'underpowered'
    if rmse <= 0.04 and (rho is not None and rho >= 0.4): return 'VALUE PRESENT (unread -> recal)'
    if (rho is None or np.isnan(rho) or rho < 0.2) and rmse > 0.06: return 'VALUE ABSENT (needs lambda*MSE)'
    return 'partial (recoverable but noisy)'

synth_row = {'cell': 'AA-enc x AA SYNTHETIC (powered)', 'n_pairs': len(synth),
             'pred_mean': float(sy_pred.mean()), 'spearman': rho_synth,
             'iso_rmse_row': rmse_synth, 'iso_rmse_grp': np.nan,
             'median_n_true': np.nan, 'frac10': np.nan, 'frac50': np.nan, 'prec10': np.nan}
table = [synth_row] + rows
dec_df = pd.DataFrame(table)

print('=' * 104)
print('COLAB19 DECISION TABLE — high-sim (>= 0.70) value fidelity + set-based oracle retrieval')
print('=' * 104)
print(f'{"cell":<34}{"n":>5}{"rho":>7}{"rmse_row":>10}{"rmse_grp":>10}{"med|T|":>8}{"frac@10":>9}  verdict')
print('-' * 104)
for d in table:
    print(f"{d['cell']:<34}{d['n_pairs']:>5}{fmt(d.get('spearman'),2):>7}"
          f"{fmt(d.get('iso_rmse_row')):>10}{fmt(d.get('iso_rmse_grp')):>10}"
          f"{fmt(d.get('median_n_true'),0):>8}{fmt(d.get('frac10')):>9}  {verdict(d)}")

dec_df.to_csv('colab19_decision_table.csv', index=False)
pd.DataFrame(per_pair).to_csv('colab19_per_pair.csv', index=False)
pd.DataFrame(synth, columns=['seq_a', 'seq_b', 'norm_lev']).assign(pred_l2sim=sy_pred).to_csv(
    'colab19_synth_predictions.csv.gz', index=False)
print('\nSaved: colab19_decision_table.csv, colab19_per_pair.csv, colab19_synth_predictions.csv.gz')
print('iso_rmse is in normLev units: 0.03 => after a monotone recalibration, predicted')
print('normLev is within +-0.03 of true. That, not retrieval, is the retrain-or-not number.')

## How to read this notebook

**The AA SYNTHETIC row is the verdict for the primary question.**
- `iso_rmse` small (<= ~0.04) + decent `rho` -> within-high-band value is *present in the
  embedding*, only the readout saturated. Fix is cheap: monotone recalibration or a small
  regression head on the **frozen** encoder. A `lambda*MSE` retrain is **not** needed for AA.
- `iso_rmse` fat (> ~0.06) + low `rho` -> CE genuinely collapsed it. Then `lambda*MSE`
  (objective lever, scoped to the high band) is justified.

**The 2x3 cells answer "why is cross-rep bad" with the value-vs-ranking dissociation.**
Compare `iso_rmse` (value) against `frac@10` (ranking-vs-oracle) and `med|T|` (crowding):
- value present + frac@10 high -> works (AA).
- value present + frac@10 not low, but `med|T|` huge -> the task is crowded, not broken;
  the encoder tracks magnitude, the alphabet is just ill-posed for single-target retrieval.
- value absent -> the signal itself is too coarse to encode (the SS noise-floor story).

`iso_rmse_grp` (component-held-out) vs `iso_rmse_row`: if grp >> row, the row-wise number
was inflated by shared proteins across folds — trust grp. Bootstrap CIs in the CSV flag the
moderate-n feeds (3Di) where point estimates are noisy.

**Next step is decided by the AA synthetic `iso_rmse`**, not assumed: small -> recalibration
/ regression readout (cheap, maybe no retrain); fat -> `lambda*MSE`. Deferred: **colab20**
full-alphabet / natural-language generality probe; **colab21** masked-adaptive-pooling
ablation (the current pool averages over zeroed PAD buckets, a length leak).
